## Quantization of the original models

### LLaVA-7b

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  3.69it/s]


Latency: 0.517096996307373 s
GPU Memory Requirement: 13610.48583984375 MiB

The


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "8"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", load_8bit=True, device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=50,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
The `load_in_4bit` and `load_in_8bit` arguments are deprecated and will be removed in the future versions. Please, pass a `BitsAndBytesConfig` object in `quantization_config` argument instead.
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  

Latency: 15.239479780197144 s
GPU Memory Requirement: 7519.68505859375 MiB

The image is funny because it features a dog wearing a Renaissance-style dress and a hat, which is an unusual and amusing sight. Dogs are not typically associated with human fashion or artistic styles, so seeing a dog dressed up


In [3]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
)[0]
torch.cuda.empty_cache()

412 ms ± 2.65 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


In [4]:
model

LlavaLlamaForCausalLM(
  (model): LlavaLlamaModel(
    (embed_tokens): Embedding(32000, 4096, padding_idx=0)
    (layers): ModuleList(
      (0-31): 32 x LlamaDecoderLayer(
        (self_attn): LlamaAttention(
          (q_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (k_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (v_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (o_proj): Linear8bitLt(in_features=4096, out_features=4096, bias=False)
          (rotary_emb): LlamaRotaryEmbedding()
        )
        (mlp): LlamaMLP(
          (gate_proj): Linear8bitLt(in_features=4096, out_features=11008, bias=False)
          (down_proj): Linear8bitLt(in_features=11008, out_features=4096, bias=False)
          (up_proj): Linear8bitLt(in_features=4096, out_features=11008, bias=False)
          (act_fn): SiLU()
        )
        (input_layernorm): LlamaRMSNorm()
        (post_attention_layernorm): LlamaRMSN

### LLaVA layerwise 

#### Original layer 5

In [2]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_5_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-5-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
# model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=10,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
Loading checkpoint shards: 100%|██████████| 2/2 [00:03<00:00,  1.61s/it]


Loading pruned model...
Deleting 5 layers from 'layers'
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Latency: 0.49121618270874023 s
GPU Memory Requirement: 18014.39794921875 MiB

The image is funny because it features a dog


In [4]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
torch.cuda.empty_cache()

88 ms ± 8.51 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)


#### Original layer 10

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_10_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-10-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.46it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Latency: 0.4599738121032715 s
GPU Memory Requirement: 9664.35107421875 MiB

The


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
torch.cuda.empty_cache()

87 ms ± 1.26 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Original layer 15

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_15_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-15-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.45it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Latency: 0.4598259925842285 s
GPU Memory Requirement: 7724.2705078125 MiB

The


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

62.4 ms ± 579 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### Original layer 20

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
torch.set_default_device(device)

tokenizer, model, image_processor, context_len = load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_21_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-21-dist-2.0-l2-0.5-layer--1-short",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.96it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Latency: 0.38382959365844727 s
GPU Memory Requirement: 5396.173828125 MiB

The


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

49.9 ms ± 638 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### int 8

#### quantized layer 5

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_5_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-5-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.18it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Converting model to 8-bit...
Loading 8-bit weights...
Latency: 2.1661617755889893 s
GPU Memory Requirement: 6473.5068359375 MiB

The image is funny because it features a dog wearing a Renaissance-


In [3]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]
torch.cuda.empty_cache()

125 ms ± 937 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### quantized layer 10

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_10_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-10-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.20it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Converting model to 8-bit...
Loading 8-bit weights...
Latency: 1.7373251914978027 s
GPU Memory Requirement: 5483.61572265625 MiB

The image is funny because it features a dog dressed in a human-


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

108 ms ± 5.85 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### quantized layer 15

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_15_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-15-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.31it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Converting model to 8-bit...
Loading 8-bit weights...
Latency: 1.5045201778411865 s
GPU Memory Requirement: 4491.724609375 MiB

The image is funny because it features a dog dressed in a human-


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

88.3 ms ± 4.93 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


#### quantized layer 21

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/ShortGPT/prune_log/llava-v1.5-7b_pruned_21_50_samples/pruned_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-21-dist-2.0-l2-0.5-layer--1-short",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  5.70it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Converting model to 8-bit...
Loading 8-bit weights...
Latency: 1.1096408367156982 s
GPU Memory Requirement: 3303.6552734375 MiB

The image is funny because the image is a picture of a dog sitting


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

60.1 ms ± 1.31 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


## Original width 0.2


In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.2_llava/pytorch_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-0.2-dist-2.0-l2-0.5-layer--1",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.42it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Latency: 1.0194275379180908 s
GPU Memory Requirement: 11530.36669921875 MiB

The image is funny because it features a dog wearing a Renaissance-


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

93.3 ms ± 606 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### Origianal width 0.4

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.4_llava/pytorch_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-0.4-dist-2.0-l2-0.5-layer--1",device=device)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.61it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Latency: 1.178985357284546 s
GPU Memory Requirement: 9548.21044921875 MiB

The image is funny because it features a dog wearing a dress,


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

80.7 ms ± 634 μs per loop (mean ± std. dev. of 7 runs, 10 loops each)


### width 0.2 int8

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.2_llava/pytorch_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-0.2-dist-2.0-l2-0.5-layer--1",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  6.34it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Converting model to 8-bit...
Loading 8-bit weights...
Latency: 2.2265071868896484 s
GPU Memory Requirement: 6388.72021484375 MiB

The image is funny because it features a dog wearing a Renaissance-


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

143 ms ± 2.54 ms per loop (mean ± std. dev. of 7 runs, 10 loops each)


### width 0.4 int8

In [1]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2"
import sys
sys.path.append("/shared-local/aoq609/VLMCompression/VLM")
from llava.model.builder import load_pruned_llava_model_all
from llava.mm_utils import tokenizer_image_token, process_images, get_model_name_from_path
import torch
from PIL import Image
import time


device = 'cuda:0'# or cpu
tokenizer, model, image_processor, context_len =load_pruned_llava_model_all("liuhaotian/llava-v1.5-7b", '/shared-local/aoq609/VLMCompression/LLM-Pruner/LLMPruner/prune_log/llava-v1.5-7b_0.4_llava/pytorch_model.bin',lora="/shared-local/aoq609/VLMCompression/VLM/llava/checkpoints/llava-v1.5-7b-0.4-dist-2.0-l2-0.5-layer--1",device=device,load_8bit=True)
prompt = 'Why is the image funny?'
text = f"A chat between a curious user and an artificial intelligence assistant. The assistant gives helpful, detailed, and polite answers to the user's questions. USER: <image>\n{prompt} ASSISTANT:"
text_chunks = [tokenizer(chunk).input_ids for chunk in text.split('<image>')]
input_ids = torch.tensor(text_chunks[0] + [-200] + text_chunks[1], dtype=torch.long).unsqueeze(0).to(device)

# image, sample images can be found in images folder
image = Image.open('/shared-local/aoq609/LLaVA_back/playground/data/eval/llava-bench-in-the-wild/images/003.jpg')
image_tensor = process_images([image], image_processor,model.config).to(dtype=model.dtype, device=device)
model.to(device)
start = time.time()
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=15,
    use_cache=True,
)[0]
end = time.time()
latency = end - start
print("Latency: {} s".format(latency))
print("GPU Memory Requirement: {} MiB\n".format(torch.cuda.memory_allocated()/1024/1024))
out=tokenizer.decode(output_ids, skip_special_tokens=True).strip()
print(out)


/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
You are using a model of type llava to instantiate a model of type llava_llama. This is not supported for all configurations of models and can yield errors.
/shared-local/aoq609/anaconda3/envs/phi/lib/python3.10/site-packages/huggingface_hub/file_download.py:1132: FutureWarning: `resume_download` is deprecated and will be removed in version 1.0.0. Downloads always resume when possible. If you want to force a new download, use `force_download=True`.
  warnings.warn(
Loading checkpoint shards: 100%|██████████| 2/2 [00:00<00:00,  4.88it/s]


Loading pruned model...
Loading LoRA weights...
Merging LoRA weights...
Model is loaded...
Converting model to 8-bit...
Loading 8-bit weights...
Latency: 2.3775973320007324 s
GPU Memory Requirement: 5389.46630859375 MiB

The image is funny because it features a dog wearing a dress,


In [2]:
%%timeit
output_ids = model.generate(
    input_ids,
    images=image_tensor,
    max_new_tokens=1,
    use_cache=True,
    repetition_penalty=1.0 # increase this to avoid chattering
)[0]

147 ms ± 6.17 ms per loop (mean ± std. dev. of 7 runs, 1 loop each)
